# CeNN V5.3 — Qwen3.5-0.8B: fidelity-guided cross calibration and paired GPU benchmarking

**Research runner · Canonical Python implementation in [`TinyCeNN-LM`](https://github.com/vtavakkoli/TinyCeNN-LM)**

**Purpose.** Your V5.2 results showed good local restoration (cosine ~0.964) but <98% next-token teacher agreement and slower end-to-end prefill. V5.3 evaluates a **query × causal-memory-output conditioned low-rank correction** with a zero-initialized warm start, training under controlled local losses, a separate *train-only* teacher KL option, and measured native-model comparisons.

**Automatic and upload-free.** Downloads the publicly available Qwen3.5-0.8B checkpoint and WikiText-2. Trains a V5-compatible baseline and then a V5.2 joint-calibrated reference from scratch; **does not pretend to restore the previously uploaded V5.2 trained weights**. Subsequent V5.3 variants warm-start from the locally trained V5.2 reference.

**Scope/limits:** layer 11 full attention, 1 exact native KV group + 1 approximate CeNN group; text-only unpadded prefill; `use_cache=False`; Python/fp32 chunkwise recurrence; no fused kernels or cached autoregressive generation. **No speed or quality claim without passing all validation and diagnostic audit gates.**


## 0 · Setup and reproducibility

Set **Runtime → Change runtime type → NVIDIA GPU**. The setup clones the current `main` branch, validates the V5.3 Python contract tests, records Git SHA, and imports the **repository-owned implementation**; this notebook contains no duplicate model source and requires no upload. Execute the cells in order. The final ZIP download contains checkpoints and results.


In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"

In [ ]:
import io, os, sys, gc, json, hashlib, subprocess, math, random, statistics, time
from pathlib import Path
from contextlib import contextmanager, nullcontext
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
                    'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))
V53_MODULE=REPO/'src/tinycenn_lm/cenn_head_rescue_v53.py'
V53_TESTS=REPO/'tests/test_cenn_head_rescue_v53.py'
assert V53_MODULE.is_file() and V53_TESTS.is_file(), 'V5.3 source/tests missing from repository main'
MODULE_SHA256=hashlib.sha256(b''.join((REPO/'src/tinycenn_lm'/f).read_bytes()
    for f in ('cenn_head_rescue_v51.py','cenn_head_rescue_v52.py',
              'cenn_head_rescue_v53.py'))).hexdigest()
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from tinycenn_lm.cenn_head_rescue_v51 import CeNNQueryRescueV51,warmstart_v5_delta_core
from tinycenn_lm.cenn_head_rescue_v52 import CeNNHeadRescueV52,warmstart_v51_to_v52
from tinycenn_lm.cenn_head_rescue_v53 import CeNNHeadRescueV53,warmstart_v52_to_v53
from transformers import AutoTokenizer, AutoConfig, Qwen3_5ForCausalLM
from huggingface_hub import model_info
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert torch.cuda.is_available(), 'A CUDA GPU is required for the Qwen3.5 comparison'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('Git revision',GIT_SHA,'Source SHA256',MODULE_SHA256)
print('GPU:',torch.cuda.get_device_name(0),'dtype:',DTYPE)
subprocess.run([sys.executable,'-m','pytest','-q',str(V53_TESTS)],check=True)


## 1 · Experiment settings

Default quick profile is intended for debugging and candidate screening, **not publication-grade evidence**. The V5 and V5.2 references are retrained automatically so no user ZIP or checkpoints are required. Increase calibration size and seeds before drawing statistical conclusions.


In [ ]:
# @title No-upload V5.3 experiment settings
PROFILE='quick' # @param ['quick','research']
MODEL_ID='Qwen/Qwen3.5-0.8B'
TARGET_LAYER=11
SEQ_LEN=256 if PROFILE=='quick' else 512
TRAIN_BLOCKS=48 if PROFILE=='quick' else 128
DEV_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_START_TOKEN=1_250_000   # disjoint from prior V5.2 1M-token audit; still train-partition diagnostic
BASELINE_STEPS=140 if PROFILE=='quick' else 500
REFERENCE_STEPS=220 if PROFILE=='quick' else 700
STEPS=180 if PROFILE=='quick' else 600
TASK_STEPS=35 if PROFILE=='quick' else 120
BATCH_SIZE=2
BASELINE_LR=2e-3
REFERENCE_LR=8e-4
LR=5e-4
CHUNK_SIZE=32
SEED=4053
RANK=16
CROSS_RANK=16
OUT=Path('/content/qwen35_cenn_head_rescue_v53')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
CANDIDATES={
 'v52_frozen_w128':dict(family='v52',window=128,mode='none'),
 'v53_cross_w128':dict(family='v53',window=128,mode='cross'),
 'v53_router_cross_w128':dict(family='v53',window=128,mode='router_cross'),
 'v53_joint_w64':dict(family='v53',window=64,mode='joint'),
 'v53_joint_w128':dict(family='v53',window=128,mode='joint'),
 'v53_joint_w256':dict(family='v53',window=256,mode='joint'),
}
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Auto-training references, no checkpoint uploads:',BASELINE_STEPS,REFERENCE_STEPS)
print('Candidates',list(CANDIDATES),'training steps',STEPS)


## 2 · Pin and load the original Qwen model

The native layer is frozen and temporarily swapped only inside explicit context managers. Check that the all-exact rescue wrapper matches the original Qwen logits before any experimental fitting.


In [ ]:
REVISION=model_info(MODEL_ID).sha
print('Pinned Qwen model revision:',REVISION)
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text' and cfg.num_hidden_layers==24 and cfg.hidden_size==1024
full=[i for i,t in enumerate(cfg.layer_types) if t=='full_attention']
assert TARGET_LAYER in full and full==[3,7,11,15,19,23],full
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False)
model.config.use_cache=False
LAYERS=model.model.layers
NATIVE=LAYERS[TARGET_LAYER].self_attn
HD=int(NATIVE.head_dim)
QHEADS=int(NATIVE.q_proj.out_features//(2*HD))
KVHEADS=int(NATIVE.k_proj.out_features//HD)
assert (QHEADS,KVHEADS,HD)==(8,2,256),(QHEADS,KVHEADS,HD)
@contextmanager
def graft(adapter):
    assert LAYERS[TARGET_LAYER].self_attn is NATIVE
    LAYERS[TARGET_LAYER].self_attn=adapter
    try:yield
    finally:LAYERS[TARGET_LAYER].self_attn=NATIVE
sample=tokenizer('CeNN is a cellular associative memory model.',return_tensors='pt',
                 add_special_tokens=False).input_ids.to(DEVICE)
all_exact=Qwen35HeadRescueV5(NATIVE,None,exact_groups=(0,1)).eval()
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    reference=model(input_ids=sample,use_cache=False,logits_to_keep=0).logits.float()
    with graft(all_exact):parity=model(input_ids=sample,use_cache=False,logits_to_keep=0).logits.float()
max_difference=(reference-parity).abs().max().item()
print('Exact-native parity:',max_difference)
assert max_difference<.05 and bool((reference.argmax(-1)==parity.argmax(-1)).all())
del all_exact,parity,reference


## 3 · Data splits and predeclared diagnostic audit

Training uses WikiText-2 train prefix; selection uses **WikiText-2 validation**. A disjoint *train tail* after 1,250,000 tokenizer tokens is evaluated once, after validation-only selection. This is an **internal diagnostic**, not an independent benchmark or a fresh external test dataset. The prior WikiText-2 test split is not consulted.


In [ ]:
from datasets import load_dataset

def build_windows(split,count,length,start_token=0):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    need=start_token+count*length
    tokens=[]
    for row in ds:
        text=row['text'].strip()
        if not text:continue
        tokens.extend(tokenizer.encode(text,add_special_tokens=False))
        tokens.append(tokenizer.eos_token_id)
        if len(tokens)>=need:break
    if len(tokens)<need:raise RuntimeError(f'{split}: {len(tokens)} tokens, need {need}')
    return torch.tensor(tokens[start_token:need],dtype=torch.long).view(count,length)
TRAIN=build_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=build_windows('validation',DEV_BLOCKS,SEQ_LEN)
# This split is not constructed before model selection. AUDIT_START_TOKEN is fixed above.
FINGERPRINTS={'train':hashlib.sha256(TRAIN.numpy().tobytes()).hexdigest(),
              'validation':hashlib.sha256(DEV.numpy().tobytes()).hexdigest()}
assert TRAIN_BLOCKS*SEQ_LEN < AUDIT_START_TOKEN
print('Train',TRAIN.shape,'Dev',DEV.shape,'Audit offset',AUDIT_START_TOKEN)

## 4. Capture **original teacher mixer** inputs and outputs once
Hook the actual full-attention module, retaining Qwen's RoPE tensors. None of this includes the previously held-out audit texts.

In [ ]:
TEACHER={}
@torch.inference_mode()
def forward(ids,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=ids.to(DEVICE),use_cache=False,logits_to_keep=keep)

def capture(name,blocks):
    cols={v:[] for v in ('x','y','cos','sin')}
    def hook(module,args,kwargs,result):
        inp=kwargs.get('hidden_states',args[0] if args else None)
        assert inp is not None
        val=result[0] if isinstance(result,tuple) else result
        cos,sin=kwargs['position_embeddings']
        cols['x'].append(inp.detach().to('cpu',dtype=torch.float16))
        cols['y'].append(val.detach().to('cpu',dtype=torch.float16))
        cols['cos'].append(cos.detach().cpu().to(torch.float32))
        cols['sin'].append(sin.detach().cpu().to(torch.float32))
    handle=NATIVE.register_forward_hook(hook,with_kwargs=True)
    try:
        for start in range(0,len(blocks),BATCH_SIZE):forward(blocks[start:start+BATCH_SIZE])
    finally:handle.remove()
    TEACHER[name]={key:torch.cat(val,0).contiguous() for key,val in cols.items()}
    print(name,{key:tuple(val.shape) for key,val in TEACHER[name].items()})
capture('train',TRAIN)
capture('dev',DEV)
torch.cuda.empty_cache()

## 5 · Automatic V5-compatible baseline

Train a Delta + Local64 core for the approximate Qwen KV group, or reuse a matching local checkpoint from the same Colab runtime. Its reproducibility fingerprint is bound to code SHA, base-model revision, data hashes and training settings. No files need to be uploaded.


In [ ]:
def teacher_batch(split,indices):
    raw=TEACHER[split]
    return (raw['x'][indices].to(DEVICE,dtype=DTYPE),
            raw['y'][indices].to(DEVICE,dtype=DTYPE),
            (raw['cos'][indices].to(DEVICE),raw['sin'][indices].to(DEVICE)))

def local_metrics(pred,target):
    a,b=pred.float(),target.float()
    return ((a-b).square().mean()/b.square().mean().clamp_min(1e-8),
            F.cosine_similarity(a,b,dim=-1).mean())

def new_v5_baseline():
    core=CeNNDeltaCellV3(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                         mode='none',local_window=64).to(DEVICE)
    return Qwen35HeadRescueV5(NATIVE,core,exact_groups=(0,)).eval()

@torch.no_grad()
def validate_baseline(adapter):
    adapter.eval()
    error=energy=cosine_sum=0.;n=0
    for begin in range(0,DEV_BLOCKS,BATCH_SIZE):
        indices=torch.arange(begin,min(begin+BATCH_SIZE,DEV_BLOCKS))
        x,y,rope=teacher_batch('dev',indices)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
        a,b=pred.float(),y.float()
        error+=(a-b).square().sum().item()
        energy+=b.square().sum().item()
        cosine_sum+=F.cosine_similarity(a,b,dim=-1).sum().item()
        n+=a.shape[0]*a.shape[1]
    return dict(nmse=error/max(energy,1.e-8),cosine=cosine_sum/n)

BOOTSTRAP_SPEC=dict(model_id=MODEL_ID,model_revision=REVISION,source_git_sha=GIT_SHA,
    source_module_sha256=MODULE_SHA256,layer=TARGET_LAYER,profile=PROFILE,
    train_hash=FINGERPRINTS['train'],dev_hash=FINGERPRINTS['validation'],
    feature_dim=64,local_window=64,seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,
    dev_blocks=DEV_BLOCKS,chunk_size=CHUNK_SIZE,baseline_steps=BASELINE_STEPS,
    batch_size=BATCH_SIZE,lr=BASELINE_LR,seed=SEED)
BOOTSTRAP_FINGERPRINT=hashlib.sha256(
    json.dumps(BOOTSTRAP_SPEC,sort_keys=True).encode()).hexdigest()
BOOTSTRAP_PATH=OUT/'checkpoints'/f'v5_auto_L{TARGET_LAYER}_delta_f64_w64.pt'

if BOOTSTRAP_PATH.exists():
    candidate=torch.load(BOOTSTRAP_PATH,map_location='cpu',weights_only=True)
    if candidate.get('bootstrap_fingerprint')==BOOTSTRAP_FINGERPRINT:
        baseline=new_v5_baseline()
        baseline.core.load_state_dict(candidate['core_state'],strict=True)
        V5_CORE_STATE={k:v.detach().cpu().clone() for k,v in candidate['core_state'].items()}
        BOOTSTRAP_VALIDATION=candidate['validation']
        del baseline
        print('Reusing locally trained baseline, matching fingerprint',BOOTSTRAP_FINGERPRINT[:12])
    else:
        print('Existing baseline settings differ; training a new baseline automatically')
        V5_CORE_STATE=None
else:
    V5_CORE_STATE=None

if V5_CORE_STATE is None:
    torch.manual_seed(SEED+TARGET_LAYER*17)
    baseline=new_v5_baseline()
    baseline.core.requires_grad_(True)
    optimizer=torch.optim.AdamW(baseline.core.parameters(),lr=BASELINE_LR,
                                weight_decay=.01)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,T_max=BASELINE_STEPS,eta_min=BASELINE_LR*.10)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    sampling=torch.randint(0,TRAIN_BLOCKS,(BASELINE_STEPS,BATCH_SIZE),
        generator=torch.Generator().manual_seed(SEED+11))
    best=float('inf');best_sd=None;best_stats=None
    for step in range(1,BASELINE_STEPS+1):
        x,y,rope=teacher_batch('train',sampling[step-1])
        baseline.train();optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=baseline(x,position_embeddings=rope)
            nmse,cosine=local_metrics(pred,y)
            loss=nmse+.2*(1.-cosine)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'Nonfinite V5 baseline loss at step {step}')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(baseline.core.parameters(),1.)
        scaler.step(optimizer);scaler.update();scheduler.step()
        if step==1 or step==BASELINE_STEPS or step%max(1,BASELINE_STEPS//8)==0:
            stats=validate_baseline(baseline)
            score=stats['nmse']+.2*(1.-stats['cosine'])
            if score<best:
                best=score;best_stats=stats
                best_sd={k:v.detach().cpu().clone()
                         for k,v in baseline.core.state_dict().items()}
            print(f'V5 auto-bootstrap {step}/{BASELINE_STEPS} '
                  f'cos={stats["cosine"]:.5f}, NMSE={stats["nmse"]:.5f}',flush=True)
    assert best_sd is not None
    V5_CORE_STATE=best_sd
    BOOTSTRAP_VALIDATION=best_stats
    torch.save(dict(core_state=best_sd,validation=best_stats,
                    bootstrap_fingerprint=BOOTSTRAP_FINGERPRINT,bootstrap_spec=BOOTSTRAP_SPEC,
                    model_revision=REVISION,variant='rescue_g0_delta_f64_w64'),
               BOOTSTRAP_PATH)
    del baseline,optimizer,scheduler,scaler,best_sd
    torch.cuda.empty_cache()
V5_CHECKPOINT_SHA256=hashlib.sha256(BOOTSTRAP_PATH.read_bytes()).hexdigest()
BOOTSTRAP_VARIANT='rescue_g0_delta_f64_w64'
assert len(V5_CORE_STATE)==len(new_v5_baseline().core.state_dict())
print('Baseline checkpoint:',BOOTSTRAP_PATH,'SHA-256:',V5_CHECKPOINT_SHA256)
print('Baseline mixer fidelity:', BOOTSTRAP_VALIDATION)


## 6 · Architecture: V5.2 calibrated seed → V5.3 cross-conditioned readout

First warm-start and train the **same V5.2 joint W128** control as the previous study. Then initialize all V5.3 variants from that checkpoint. The new cross-readout consumes the current query and the already causal local/recurrent output. Its output weights and calibration parameters are initialized to zero, giving exact warm-start parity at the same window length.


In [ ]:
def make_v52_reference(window=128):
    old=CeNNQueryRescueV51(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                           local_window=window,correction_rank=RANK,
                           query_router=True,query_correction=True)
    warmstart_v5_delta_core(old,V5_CORE_STATE)
    core=CeNNHeadRescueV52(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
             local_window=window,correction_rank=RANK,query_router=True,
             query_correction=True,post_rank=RANK,post_correction=True)
    warmstart_v51_to_v52(core,old.state_dict())
    return Qwen35HeadRescueV5(NATIVE,core.to(DEVICE),exact_groups=(0,)).eval()

def base_variant(name):
    return 'v53_joint_w128' if name=='v53_joint_w128_plus_kl' else name

def build_candidate(name):
    conf=CANDIDATES[base_variant(name)]
    if conf['family']=='v52':
        adapter=make_v52_reference(conf['window'])
        adapter.core.load_state_dict(V52_CORE_STATE,strict=True)
        return adapter
    core=CeNNHeadRescueV53(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
         local_window=conf['window'],correction_rank=RANK,query_router=True,
         query_correction=True,post_rank=RANK,post_correction=True,
         cross_rank=CROSS_RANK,cross_correction=True,output_calibration=True)
    warmstart_v52_to_v53(core,V52_CORE_STATE)
    return Qwen35HeadRescueV5(NATIVE,core.to(DEVICE),exact_groups=(0,)).eval()


## 7 · Train V5.2 reference and V5.3 candidates

**Training policy:** same frozen Qwen projections and teacher mixer activations; native exact group untouched. Local fit minimizes normalized MSE plus cosine discrepancy. Controlled modes train only new cross/calibration parameters, add router parameters, or jointly fine-tune the approximate group. Candidates begin from identical V5.2 reference weights, except that changing local-window size changes which tokens are retrieved. Validation selects the best checkpoint for each candidate.


In [ ]:
def validate_local(adapter):
    adapter.eval();err=energy=cos_sum=0.;count=0
    with torch.no_grad():
        for begin in range(0,DEV_BLOCKS,BATCH_SIZE):
            idx=torch.arange(begin,min(DEV_BLOCKS,begin+BATCH_SIZE))
            x,y,rope=teacher_batch('dev',idx)
            with torch.autocast('cuda',dtype=DTYPE):p,_=adapter(x,position_embeddings=rope)
            pp,yy=p.float(),y.float()
            err+=(pp-yy).square().sum().item();energy+=yy.square().sum().item()
            cos_sum+=F.cosine_similarity(pp,yy,dim=-1).sum().item()
            count+=pp.shape[0]*pp.shape[1]
    return dict(cosine=cos_sum/count,nmse=err/max(energy,1e-8))

def local_loss(pred,target):
    p,y=pred.float(),target.float()
    return (p-y).square().mean()/y.square().mean().clamp_min(1e-8) + .2*(1-F.cosine_similarity(p,y,dim=-1).mean())

def tune_local(adapter,params,steps,lr,seed,name):
    if not params:
        return adapter,validate_local(adapter),{k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
    opt=torch.optim.AdamW(params,lr=lr,weight_decay=.005)
    schedule=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=steps,eta_min=lr*.1)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    best=math.inf;best_state=None;best_stats=None
    inds=torch.randint(0,TRAIN_BLOCKS,(steps,BATCH_SIZE),generator=torch.Generator().manual_seed(seed))
    for step in range(1,steps+1):
        x,y,rope=teacher_batch('train',inds[step-1])
        adapter.train();opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            prediction,_=adapter(x,position_embeddings=rope)
            loss=local_loss(prediction,y)
        if not bool(torch.isfinite(loss)):raise FloatingPointError(f'{name} step {step}: nonfinite loss')
        scaler.scale(loss).backward();scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params,1.)
        scaler.step(opt);scaler.update();schedule.step()
        if step==1 or step==steps or step%max(1,steps//7)==0:
            stats=validate_local(adapter)
            score=stats['nmse']+.2*(1-stats['cosine'])
            if score<best:
                best=score;best_stats=stats
                best_state={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
            print(f'{name} {step}/{steps}: cosine={stats["cosine"]:.5f}, NMSE={stats["nmse"]:.5f}',flush=True)
    assert best_state is not None
    adapter.core.load_state_dict(best_state);adapter.eval()
    return adapter,best_stats,best_state

REFERENCE_SPEC=dict(model_revision=REVISION,repo_sha=GIT_SHA,baseline_sha=V5_CHECKPOINT_SHA256,
    source_sha=MODULE_SHA256,steps=REFERENCE_STEPS,seed=SEED,seq_len=SEQ_LEN,
    train_hash=FINGERPRINTS['train'],dev_hash=FINGERPRINTS['validation'],lr=REFERENCE_LR)
REFERENCE_FINGERPRINT=hashlib.sha256(json.dumps(REFERENCE_SPEC,sort_keys=True).encode()).hexdigest()
REFERENCE_PATH=OUT/'checkpoints'/f'v52_auto_joint_L{TARGET_LAYER}_w128.pt'
V52_CORE_STATE=None
if REFERENCE_PATH.exists():
    chk=torch.load(REFERENCE_PATH,map_location='cpu',weights_only=True)
    if chk.get('fingerprint')==REFERENCE_FINGERPRINT:
        V52_CORE_STATE=chk['core_state']
        print('Reusing validated local V5.2 reference',REFERENCE_FINGERPRINT[:12])
if V52_CORE_STATE is None:
    teacher_adapter=make_v52_reference(128)
    teacher_adapter.core.requires_grad_(True)
    torch.manual_seed(SEED+43)
    teacher_adapter,reference_local,V52_CORE_STATE=tune_local(
       teacher_adapter,list(teacher_adapter.core.parameters()),REFERENCE_STEPS,
       REFERENCE_LR,SEED+100,'V5.2 reference')
    torch.save({'core_state':V52_CORE_STATE,'validation':reference_local,
                'fingerprint':REFERENCE_FINGERPRINT,'config':teacher_adapter.core.config},REFERENCE_PATH)
    del teacher_adapter;torch.cuda.empty_cache()
REFERENCE_SHA256=hashlib.sha256(REFERENCE_PATH.read_bytes()).hexdigest()

def set_trainables(adapter,name):
    adapter.core.requires_grad_(False)
    mode=CANDIDATES[base_variant(name)]['mode']
    if mode=='none':return []
    cross={'cross_down','cross_up','cross_log_gain','output_log_scale','output_bias'}
    router={'local_logits','router_w','router_bias'}
    allowed=cross if mode=='cross' else cross|router if mode=='router_cross' else None
    for key,p in adapter.core.named_parameters():
        if allowed is None or key in allowed:p.requires_grad_(True)
    enabled=[p for p in adapter.core.parameters() if p.requires_grad]
    assert enabled and not any(p.requires_grad for p in NATIVE.parameters())
    return enabled

def fit(variant):
    torch.manual_seed(SEED+TARGET_LAYER*101)
    adapter=build_candidate(variant)
    params=set_trainables(adapter,variant)
    return tune_local(adapter,params,STEPS,LR,SEED+777,variant)

# Verify exact output parity after strict V5.2→V5.3 warm-start (same local window).
with torch.inference_mode():
    x,y,rope=teacher_batch('dev',torch.tensor([0]))
    with torch.autocast('cuda',dtype=DTYPE):
        v52,_=build_candidate('v52_frozen_w128')(x,position_embeddings=rope)
        v53,_=build_candidate('v53_cross_w128')(x,position_embeddings=rope)
    warmstart_difference=float((v52-v53).abs().max())
    assert warmstart_difference<.005,warmstart_difference
print('V5.2 to V5.3 zero-output warm-start difference:',warmstart_difference)
del x,y,rope,v52,v53;torch.cuda.empty_cache()


## 8 · Whole-Qwen validation and paired GPU timing

Evaluate **every next-token position**. Paired timing alternates native and candidate execution on the same inputs, dtype and device, excluding cold-start and training overhead. This measures full-model `use_cache=False` prefill; it does **not** establish cached decoding speed.


In [ ]:
@torch.inference_mode()
def measure_lm(blocks,reference=None):
    count=0;loss_total=0.;correct=0;agreement=0;top1=[]
    for begin in range(0,len(blocks),BATCH_SIZE):
        tok=blocks[begin:begin+BATCH_SIZE]
        idx=torch.arange(tok.shape[1]-1,device=DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            logits=model(input_ids=tok.to(DEVICE),use_cache=False,
                         logits_to_keep=idx).logits.float()
        gold=tok[:,1:].to(DEVICE)
        logp=F.log_softmax(logits,dim=-1)
        loss_total-=logp.gather(-1,gold.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        correct+=(pred==gold).sum().item();count+=gold.numel()
        if reference is None:top1.append(pred.detach().cpu().to(torch.int32))
        else:
            ref=reference[begin//BATCH_SIZE].to(DEVICE)
            assert ref.shape==pred.shape
            agreement+=(ref==pred).sum().item()
    result=dict(nll=loss_total/count,accuracy=correct/count,decisions=count)
    if reference is not None:result['teacher_agreement']=agreement/count
    return result,top1

@torch.inference_mode()
def paired_timing(adapter,blocks,warmups=3,repeats=12):
    sample=blocks[:min(len(blocks),BATCH_SIZE)].to(DEVICE)
    native_times=[];student_times=[]
    def once(use_student):
        with graft(adapter) if use_student else nullcontext():
            begin=torch.cuda.Event(enable_timing=True)
            end=torch.cuda.Event(enable_timing=True)
            begin.record()
            with torch.autocast('cuda',dtype=DTYPE):
                logits=model(input_ids=sample,use_cache=False,logits_to_keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(logits).all())
            return begin.elapsed_time(end)
    torch.cuda.synchronize()
    for turn in range(warmups+repeats):
        if turn%2: st=once(True);nt=once(False)
        else: nt=once(False);st=once(True)
        if turn>=warmups:native_times.append(nt);student_times.append(st)
    return dict(native_ms=float(np.median(native_times)),student_ms=float(np.median(student_times)),
                speedup=float(np.median(native_times)/np.median(student_times)),
                native_iqr_ms=float(np.subtract(*np.percentile(native_times,[75,25]))),
                student_iqr_ms=float(np.subtract(*np.percentile(student_times,[75,25]))),
                repeats=repeats)

NATIVE_DEV,NATIVE_DEV_TOP1=measure_lm(DEV)
print('Native validation',NATIVE_DEV)


## 9 · Fit candidate variants, save each checkpoint, and rank on validation

Six variants compare whether the query×memory readout improves restoration over the V5.2 reference, and whether local-window 64/128/256 influences the fidelity–speed tradeoff. All variants use the same data and seed. No diagnostic audit is read here.


In [ ]:
REPORT=[]
RUN_SPEC=dict(model_revision=REVISION,git_sha=GIT_SHA,code_sha256=MODULE_SHA256,
      v5_checkpoint_sha256=V5_CHECKPOINT_SHA256,v52_reference_sha256=REFERENCE_SHA256,
      v52_reference_fingerprint=REFERENCE_FINGERPRINT,profile=PROFILE,
      seed=SEED,steps=STEPS,task_steps=TASK_STEPS,seq_len=SEQ_LEN,
      audit_offset=AUDIT_START_TOKEN,lr=LR,rank=RANK,cross_rank=CROSS_RANK,
      candidates=CANDIDATES)
FINGERPRINT=hashlib.sha256(json.dumps(RUN_SPEC,sort_keys=True).encode()).hexdigest()
for variant in CANDIDATES:
    print('\n==========',variant,'==========',flush=True)
    adapter,local,sd=fit(variant)
    with graft(adapter):dev,_=measure_lm(DEV,reference=NATIVE_DEV_TOP1)
    timing=paired_timing(adapter,DEV,repeats=7,warmups=3)
    row=dict(variant=variant,layer=TARGET_LAYER,steps=0 if variant=='v52_frozen_w128' else STEPS,
             local_dev_cosine=local['cosine'],local_dev_nmse=local['nmse'],
             dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
             dev_agreement=dev['teacher_agreement'],decisions=dev['decisions'],
             parameters=sum(p.numel() for p in adapter.core.parameters()),
             paired_speedup=timing['speedup'],native_ms=timing['native_ms'],
             student_ms=timing['student_ms'])
    REPORT.append(row)
    torch.save(dict(core_state=sd,variant=variant,run_fingerprint=FINGERPRINT,
                    source_model_revision=REVISION,base_reference_sha256=REFERENCE_SHA256,
                    core_config=adapter.core.config),
               OUT/'checkpoints'/f'v53_L{TARGET_LAYER}_{variant}.pt')
    pd.DataFrame(REPORT).to_csv(OUT/'validation_candidates.csv',index=False)
    print(json.dumps(row,indent=2),flush=True)
    del adapter,sd;gc.collect();torch.cuda.empty_cache()
print('Completed',len(REPORT),'candidate variants')


## 10 · Validation-only candidate selection

Predeclared gates: local cosine ≥0.95, teacher top-1 agreement ≥0.98, validation ΔNLL ≤+0.10. Of qualified models, prioritize speed; otherwise select a *diagnostic only* candidate by highest teacher agreement, not by a tiny incidental NLL difference. **No inference-speed success is claimed unless the final audit also passes.**


In [ ]:
MIN_COSINE=.95
MIN_AGREEMENT=.98
MAX_DELTA_NLL=.10

def select_on_validation(records):
    data=pd.DataFrame(records)
    data['quality_gate']=((data.local_dev_cosine>=MIN_COSINE)&
                          (data.dev_agreement>=MIN_AGREEMENT)&
                          (data.dev_delta_nll<=MAX_DELTA_NLL))
    accepted_rows=data[data.quality_gate]
    if len(accepted_rows):
        pick=accepted_rows.sort_values(['paired_speedup','dev_agreement','dev_delta_nll'],
                                        ascending=[False,False,True]).iloc[0]
        gate=True
    else:
        pick=data.sort_values(['dev_agreement','local_dev_cosine','dev_delta_nll'],
                              ascending=[False,False,True]).iloc[0]
        gate=False
    return data.sort_values(['quality_gate','dev_agreement','local_dev_cosine'],
                            ascending=[False,False,False]).reset_index(drop=True),pick,gate

ranked,winner,accepted=select_on_validation(REPORT)
display(ranked[['variant','local_dev_cosine','local_dev_nmse','dev_delta_nll',
                'dev_agreement','paired_speedup','quality_gate']])
print('Selected validation diagnostic:',winner.variant,'quality gate accepted?',accepted)
ranked.to_csv(OUT/'validation_ranked.csv',index=False)


## 11 · Optional train-only task-logit distillation

Fine-tune the joint W128 candidate with native-teacher KL **only on WikiText-2 train samples**, then redo validation selection. The previously uploaded V5.2 run saw a slight regression with KL, so this option is **off by default** and must earn its improvement on validation. Student checkpoint restoration uses the actual trained variant and architecture flags.


In [ ]:
# @title Optional train-only end-to-end KL alignment
ENABLE_TASK_KL=False # @param {type:'boolean'}
if ENABLE_TASK_KL:
    name='v53_joint_w128'
    path=OUT/'checkpoints'/f'v53_L{TARGET_LAYER}_{name}.pt'
    payload=torch.load(path,map_location='cpu',weights_only=True)
    assert payload['run_fingerprint']==FINGERPRINT
    adapter=build_candidate(name)
    adapter.core.load_state_dict(payload['core_state'],strict=True)
    params=set_trainables(adapter,name)
    opt=torch.optim.AdamW(params,lr=2e-5,weight_decay=.001)
    rng=np.random.default_rng(SEED+31)
    positions=torch.arange(max(0,SEQ_LEN-33),SEQ_LEN-1,2,device=DEVICE)
    for step in range(TASK_STEPS):
        choices=torch.tensor(rng.choice(len(TRAIN),BATCH_SIZE,replace=False))
        batch=TRAIN[choices].to(DEVICE)
        with torch.no_grad(),torch.autocast('cuda',dtype=DTYPE):
            teacher_logits=model(input_ids=batch,use_cache=False,
                                 logits_to_keep=positions).logits.float().detach()
        opt.zero_grad(set_to_none=True)
        with graft(adapter),torch.autocast('cuda',dtype=DTYPE):
            student_logits=model(input_ids=batch,use_cache=False,
                                 logits_to_keep=positions).logits.float()
        tprob=F.log_softmax(teacher_logits,dim=-1).exp()
        slog=F.log_softmax(student_logits,dim=-1)
        loss=F.kl_div(slog,tprob,reduction='none').sum(-1).mean()
        assert bool(torch.isfinite(loss))
        loss.backward();torch.nn.utils.clip_grad_norm_(params,1.);opt.step()
        if (step+1)%max(1,TASK_STEPS//5)==0:print('train-only KL',step+1,float(loss))
        del student_logits,teacher_logits,loss
    adapter.eval()
    local=validate_local(adapter)
    with graft(adapter):dev,_=measure_lm(DEV,reference=NATIVE_DEV_TOP1)
    timing=paired_timing(adapter,DEV,repeats=7,warmups=3)
    new=dict(variant='v53_joint_w128_plus_kl',layer=TARGET_LAYER,steps=STEPS+TASK_STEPS,
             local_dev_cosine=local['cosine'],local_dev_nmse=local['nmse'],
             dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
             dev_agreement=dev['teacher_agreement'],decisions=dev['decisions'],
             parameters=sum(p.numel() for p in adapter.core.parameters()),
             paired_speedup=timing['speedup'],native_ms=timing['native_ms'],
             student_ms=timing['student_ms'])
    REPORT.append(new)
    torch.save(dict(core_state={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()},
                    variant='v53_joint_w128_plus_kl',run_fingerprint=FINGERPRINT,
                    source_model_revision=REVISION,base_reference_sha256=REFERENCE_SHA256,
                    core_config=adapter.core.config),
               OUT/'checkpoints'/f'v53_L{TARGET_LAYER}_v53_joint_w128_plus_kl.pt')
    ranked,winner,accepted=select_on_validation(REPORT)
    ranked.to_csv(OUT/'validation_ranked.csv',index=False)
    print('Updated validation winner:',winner.variant,'eligible:',accepted)
else:
    print('Optional task KL disabled; validation selection unchanged')


## 12 · One delayed, disjoint-train-tail audit and paired full-model timing

The audit comes from a predeclared later section of WikiText-2 train, never from the prior WikiText-2 test split. This is a **diagnostic**, not a fresh held-out real-world dataset. Final pass needs validation fidelity AND audit teacher agreement ≥0.98, non-regressing loss within declared bound, and speedup >1.05×.


In [ ]:
SELECTED=str(winner.variant)
checkpoint=OUT/'checkpoints'/f'v53_L{TARGET_LAYER}_{SELECTED}.pt'
payload=torch.load(checkpoint,map_location='cpu',weights_only=True)
assert payload['run_fingerprint']==FINGERPRINT
student=build_candidate(SELECTED)
student.core.load_state_dict(payload['core_state'],strict=True);student.eval()
AUDIT=build_windows('train',AUDIT_BLOCKS,SEQ_LEN,start_token=AUDIT_START_TOKEN)
FINGERPRINTS['diagnostic_train_tail']=hashlib.sha256(AUDIT.numpy().tobytes()).hexdigest()
assert len(set(FINGERPRINTS.values()))==3
NATIVE_AUDIT,NATIVE_AUDIT_TOP1=measure_lm(AUDIT)
with graft(student):STUDENT_AUDIT,_=measure_lm(AUDIT,reference=NATIVE_AUDIT_TOP1)
PAIRED_AUDIT=paired_timing(student,AUDIT,repeats=20,warmups=5)
DELTA_NLL=STUDENT_AUDIT['nll']-NATIVE_AUDIT['nll']
FINAL_PASS=bool(accepted and DELTA_NLL<=MAX_DELTA_NLL and
                STUDENT_AUDIT['teacher_agreement']>=MIN_AGREEMENT and
                PAIRED_AUDIT['speedup']>1.05)
FINAL=dict(architecture='CeNN V5.3 query-memory output cross-calibration',
           model=MODEL_ID,model_revision=REVISION,repo_sha=GIT_SHA,
           baseline_origin='automatically_trained_no_upload',
           baseline_variant=BOOTSTRAP_VARIANT,baseline_steps=BASELINE_STEPS,
           baseline_fingerprint=BOOTSTRAP_FINGERPRINT,
           baseline_checkpoint_sha256=V5_CHECKPOINT_SHA256,
           v52_reference_sha256=REFERENCE_SHA256,v52_reference_steps=REFERENCE_STEPS,
           code_sha256=MODULE_SHA256,profile=PROFILE,run_fingerprint=FINGERPRINT,
           seed=SEED,selected=SELECTED,validation_accepted=accepted,final_pass=FINAL_PASS,
           acceptance=dict(min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
                           max_delta_nll=MAX_DELTA_NLL,min_speedup=1.05),
           validation=ranked.to_dict('records'),native_audit=NATIVE_AUDIT,
           student_audit=STUDENT_AUDIT,audit_delta_nll=DELTA_NLL,
           paired_audit=PAIRED_AUDIT,fingerprints=FINGERPRINTS,
           provenance='Internal diagnostic: disjoint WikiText-2 train tail from token offset 1250000; '
                      'prior WikiText-2 test split was not reused',
           limitations='Text-only no-cache full-prefix CUDA prefill; no fused recurrent kernel, '
                       'cached autoregressive decode, padded inputs, multimodal or external replication.')
(OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))
print(json.dumps({key:FINAL[key] for key in
    ['selected','validation_accepted','final_pass','native_audit','student_audit',
     'audit_delta_nll','paired_audit']},indent=2))
if not FINAL_PASS:print('REJECT: keep native full-attention layer 11; no speed/quality improvement claimed.')


## 13 · Optional context-length latency sweep

The selective rescue and eager fp32 recurrence have nontrivial overhead. Native attention can be faster, especially on short contexts. This optional sweep measures actual end-to-end prefill rather than extrapolating speed from small tensor operations.


In [ ]:
# @title Optional length scaling
RUN_LENGTH_SWEEP=False # @param {type:'boolean'}
SWEEP=[]
if RUN_LENGTH_SWEEP:
    for length in [128,256,512,1024,2048]:
        try:
            sample=build_windows('validation',4,length)
            timing=paired_timing(student,sample,repeats=10,warmups=4)
            SWEEP.append(dict(length=length,**timing))
            print('Length',length,timing)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache();print('OOM at',length,'stopping');break
    pd.DataFrame(SWEEP).to_csv(OUT/'context_length_timing.csv',index=False)
    FINAL['length_sweep']=SWEEP
    (OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))

## 14 · Export reproducible results

Export *all* validation rows, selected final audit, the baseline and V5.2 reference checkpoints, trained V5.3 adapters, source fingerprints, and a quality plot as one ZIP archive. The archive is generated locally in the Colab session and can be downloaded without input uploads.


In [ ]:
import shutil
fig,ax=plt.subplots(figsize=(11,5))
frame=pd.DataFrame(REPORT)
ax.barh(frame['variant'],frame['dev_agreement'])
ax.axvline(MIN_AGREEMENT,ls='--',color='gray',label='0.98 agreement gate')
ax.set(xlim=(0.94,1.0),xlabel='Validation teacher top-1 agreement',
       title='Qwen3.5 CeNN V5.3 — fidelity recovery')
ax.legend();fig.tight_layout();fig.savefig(OUT/'teacher_agreement_comparison.png',dpi=160)
(OUT/'manifest.json').write_text(json.dumps(dict(
    model=MODEL_ID,revision=REVISION,git_sha=GIT_SHA,
    notebook='Qwen35_08B_CeNN_HeadRescue_V53_CrossCalibrated_Colab.ipynb',
    profile=PROFILE,source_code_sha256=MODULE_SHA256,
    baseline_origin='automatic_training_no_upload',
    v5_baseline_sha=V5_CHECKPOINT_SHA256,v52_reference_sha=REFERENCE_SHA256,
    candidates=list(CANDIDATES),audit_split='internal WikiText-2 train tail offset 1250000',
    quality_acceptance=accepted,final_pass=FINAL_PASS),indent=2))
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Result archive:',archive)
from IPython.display import display,FileLink
display(FileLink(archive))


### Interpretation: what would qualify as progress?

- **Warm-start:** V5.3 must reproduce the V5.2 joint reference before training at the same local window, including full original Qwen attention-group 0.
- **Cross correction vs joint:** If `v53_cross_w128` improves teacher agreement over `v52_frozen_w128`, query×memory output correction adds value with minimal additional trainable parameters; compare against `v53_joint_w128` to isolate the effect of training more parameters.
- **Window ablation:** 64, 128 and 256 local windows test whether better retrieval fidelity is worth its latency and memory cost. These options are not assumed faster than native Qwen.
- **Quality gate:** cosine ≥0.95, teacher agreement ≥0.98 and ΔNLL ≤+0.10, on validation; maintain audit teacher agreement ≥0.98 and >1.05× paired speedup for final pass.
- **No cheating:** The diagnostic train tail is not a blinded external holdout. The same pretrained Qwen checkpoint and teacher are used across candidates. Single-seed, short-context screening does not prove novelty or generalization.
- **Next work:** speed needs a fused recurrent/CUDA or Triton core and hybrid cache integration, plus longer-context and multi-seed evaluations. Output correction alone adds work and cannot guarantee a performance gain.
